In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../train.csv")

X = df.drop(columns=["Price_INR_Lakhs", "Property_ID"])
y = df["Price_INR_Lakhs"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42
)

categorical_features = [
    "City", "Locality_Type", "Property_Type", "Furnishing_Status"
]

numerical_features = [
    "BHK", "Bathrooms", "Super_Area_SqFt", "Carpet_Area_SqFt",
    "Floor_Number", "Total_Floors", "Age_of_Property",
    "Parking", "Lift_Available", "Gated_Community",
    "Distance_to_Metro_km", "Distance_to_City_Center_km"
]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)],
    sparse_threshold=0
)

### Error analysis of the tuned model

In [2]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", HistGradientBoostingRegressor(
        learning_rate=0.05, max_iter=500, max_leaf_nodes=15,
        min_samples_leaf=100, l2_regularization=1.0, random_state=42
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_val)

errors = X_val.copy()
errors["actual"] = y_val.values
errors["predicted"] = y_pred
errors["abs_error"] = (errors["actual"] - errors["predicted"]).abs()

errors["price_band"] = pd.qcut(errors["actual"], q=5,
                               labels=["very low", "low", "mid", "high", "very high"])

errors.groupby("price_band", observed=True).agg(
    houses=("actual", "size"),
    avg_price=("actual", "mean"),
    MAE=("abs_error", "mean")
).round(2)

,houses,avg_price,MAE
price_band,,,
very low,3200,32.00,10.86
low,3201,62.19,17.02
mid,3199,97.42,21.48
high,3200,156.27,34.29
very high,3200,366.06,79.27


In [3]:
errors.groupby("City").agg(
    houses=("actual", "size"),
    avg_price=("actual", "mean"),
    MAE=("abs_error", "mean")
).sort_values("MAE", ascending=False).round(2)

,houses,avg_price,MAE
City,,,
Mumbai,1578,316.70,62.67
Delhi,1617,228.66,55.37
Bangalore,1656,150.06,33.00
Pune,1580,132.18,30.01
Chennai,1613,121.52,29.86
Hyderabad,1615,114.02,26.91
Chandigarh,1594,125.96,26.73
Kolkata,1549,99.58,25.14
Jaipur,1621,62.80,18.12
